# PyTorch Autograd Internals: Computation Graphs, retain_graph, grad_fn Chain, and detach

> This notebook contains all code examples from the blog post.
> [Read the full post on BotMartz](https://blog.botmartz.com/pytorch_2)

**Author:** Soham Sharma · blog.botmartz.com

In [ ]:
!pip install -q torch torchvision torchaudio

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x * y       # MulBackward0
w = z + x       # AddBackward0
out = w ** 2    # PowBackward0

print(f"out.grad_fn:           {out.grad_fn}")
print(f"out.grad_fn.next_functions: {out.grad_fn.next_functions}")

In [ ]:
import torch

def walk_graph(tensor, depth=0):
    if tensor.grad_fn is None:
        print("  " * depth + f"Leaf: {tensor}")
        return
    print("  " * depth + f"{tensor.grad_fn.__class__.__name__}")
    for parent, _ in tensor.grad_fn.next_functions:
        if parent is not None:
            walk_graph_fn(parent, depth + 1)

def walk_graph_fn(fn, depth=0):
    print("  " * depth + f"{fn.__class__.__name__}")
    for parent, _ in fn.next_functions:
        if parent is not None:
            walk_graph_fn(parent, depth + 1)

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x * y
w = z + x
out = w ** 2

walk_graph(out)

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

# out = (x*y + x)^2 = (2*3 + 2)^2 = 64
z = x * y
w = z + x
out = w ** 2

print(f"out = {out.item()}")

out.backward()

# d(out)/dx = 2*(x*y+x) * (y+1) = 2*8*4 = 64
# d(out)/dy = 2*(x*y+x) * x    = 2*8*2 = 32
print(f"x.grad = {x.grad}")
print(f"y.grad = {y.grad}")

In [ ]:
import torch

x = torch.tensor(2.0, requires_grad=True)
out = x ** 3

out.backward()  # first call — works fine
print(f"x.grad = {x.grad}")

try:
    out.backward()  # second call — graph is already freed
except RuntimeError as e:
    print(f"RuntimeError: {e}")

In [ ]:
import torch
import torch.nn as nn

# Shared encoder
encoder = nn.Linear(10, 5)
x = torch.randn(4, 10)
features = encoder(x)

# Two task heads
head_a = nn.Linear(5, 2)
head_b = nn.Linear(5, 3)

out_a = head_a(features)
out_b = head_b(features)

# Two separate losses
loss_a = out_a.mean()
loss_b = out_b.mean()

# Must retain_graph for first backward because features is used by both
loss_a.backward(retain_graph=True)
loss_b.backward()  # no retain needed for the last backward

print(f"encoder bias grad: {encoder.bias.grad}")

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 3  # dy/dx = 3x^2 = 27

# First backward — retain graph for second-order computation
grad_x = torch.autograd.grad(y, x, create_graph=True)[0]
print(f"dy/dx = {grad_x.item()}")  # 27

# Second backward — computes d²y/dx²
grad_x2 = torch.autograd.grad(grad_x, x)[0]
print(f"d²y/dx² = {grad_x2.item()}")  # 6x = 18

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 1)
x = torch.randn(32, 10)
y = torch.randn(32, 1)

# WRONG — retain_graph=True in a loop
for i in range(5):
    pred = model(x)
    loss = ((pred - y) ** 2).mean()
    loss.backward(retain_graph=True)  # graph accumulates!
    print(f"Step {i}: loss={loss.item():.4f}, graph retained")

In [ ]:
import torch
import torch.nn as nn

encoder = nn.Linear(10, 5)
decoder = nn.Linear(5, 10)

x = torch.randn(4, 10)
encoded = encoder(x)

# Detach: decoder receives the values but not the gradient
encoded_detached = encoded.detach()

reconstructed = decoder(encoded_detached)
loss = ((reconstructed - x) ** 2).mean()
loss.backward()

print(f"decoder bias grad: {decoder.bias.grad is not None}")
print(f"encoder bias grad: {encoder.bias.grad}")   # None — gradient was cut

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 1)
x = torch.randn(4, 10)

# no_grad: no gradient tracking for the entire block
with torch.no_grad():
    out = model(x)
    print(f"out.requires_grad (no_grad): {out.requires_grad}")
    print(f"out.grad_fn: {out.grad_fn}")

# detach: creates a view with gradient tracking removed
out_live = model(x)
out_detached = out_live.detach()
print(f"out_live.requires_grad: {out_live.requires_grad}")
print(f"out_detached.requires_grad: {out_detached.requires_grad}")

In [ ]:
import torch

x = torch.tensor(1.0, requires_grad=True)

for i in range(3):
    y = x * (i + 1)
    y.backward()
    print(f"After step {i}: x.grad = {x.grad}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

model = nn.Linear(5, 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)
x = torch.randn(8, 5)
y = torch.randn(8, 1)

for step in range(3):
    optimizer.zero_grad()           # ← clear accumulated gradients
    pred = model(x)
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    optimizer.step()
    print(f"Step {step}: loss={loss.item():.4f}, bias.grad={model.bias.grad.item():.4f}")